# Imagery Insights Full Scene: House image discovery

#### This notebook clusters multiple houses and try to find all images of the house across all published images
#### It then also calculcated cost


This notebook uses BigQuery and Gemini AI to spatially filter, visually analyze, and cluster street-view imagery to identify and group multiple views of a specific property.

### ⚙️ Step 1: Setup & Authentication
In this section, we install necessary libraries, authenticate with Google Cloud, and initialize the BigQuery and Gemini (Vertex AI) clients. We also set up a `CostTracker` to monitor token usage and estimated costs.

In [1]:
# @title ⚙️ 1. Setup & Authentication
# @markdown This cell installs dependencies, authenticates, and initializes the clients.

import sys
import math
import json
import requests
from google.cloud import bigquery
import geopy.distance
from google import genai
from google.genai import types
from IPython.display import JSON, display, Image as IPImage

# @markdown ### Project Settings
PROJECT_ID = "imagery-insights-sandbox" # @param {type:"string"}
DATASET_ID = "home_depot_full_scene" # @param {type:"string"}
LOCATION = "global" # @param {type:"string"}

# --- AUTH & INITIALIZATION ---
try:
    from google.colab import auth
    auth.authenticate_user()
except Exception as e:
    print("Note: Using default environment credentials.")

client_bq = bigquery.Client(project=PROJECT_ID)
client_genai = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

# --- COST TRACKING UTILITIES ---
class CostTracker:
    def __init__(self):
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cost_usd = 0.0
        self.pricing = {
            "gemini-3.5-flash": {"input": 0.50, "output": 3.00},
            "gemini-3-pro-preview": {"input": 2.00, "output": 12.00},
            "gemini-3-flash-preview": {"input": 0.50, "output": 3.00}
        }

    def add_usage(self, model_id, usage):
        if not usage: return
        in_tokens = usage.prompt_token_count
        out_tokens = usage.candidates_token_count
        self.total_input_tokens += in_tokens
        self.total_output_tokens += out_tokens
        rates = self.pricing.get(model_id, {"input": 0, "output": 0})
        cost = (in_tokens / 1_000_000 * rates["input"]) + (out_tokens / 1_000_000 * rates["output"])
        self.total_cost_usd += cost

    def print_summary(self):
        print("\n" + "="*40)
        print("💰 VERTEX AI & GEMINI USAGE SUMMARY")
        print("="*40)
        print(f"Total Input Tokens:  {self.total_input_tokens:,}")
        print(f"Total Output Tokens: {self.total_output_tokens:,}")
        print(f"Total Estimated Cost: ${self.total_cost_usd:.4f} USD")
        print("="*40)

tracker = CostTracker()

Note: Using default environment credentials.


### 📐 Step 2: Spatial Utility Functions
These helper functions handle the geometric calculations needed to determine if an image is actually looking at our target house. We calculate the 'bearing' (direction) from the camera to the target and check if it falls within the camera's Field of View (FOV).

In [2]:
# @title 📐 2. Spatial Utility Functions
def calculate_bearing(start_cw, end_cw):
    lat1, lat2 = math.radians(start_cw[0]), math.radians(end_cw[0])
    diffLong = math.radians(end_cw[1] - start_cw[1])
    x = math.sin(diffLong) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - (math.sin(lat1) * math.cos(lat2) * math.cos(diffLong))
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def is_within_fov(obs_heading, target_bearing, fov_threshold=50):
    diff = abs(obs_heading - target_bearing)
    diff = min(diff, 360 - diff)
    return diff <= fov_threshold

### 📡 Step 3: BigQuery Data Retrieval
This cell queries our BigQuery dataset to find all vehicle 'tracks' and image 'observations' near the target coordinates. It filters the results based on distance and camera orientation to ensure we only process relevant images.

In [3]:
# @title 📡 3. BigQuery Data Retrieval
# @markdown Fetches street view images near the target coordinates.

# @markdown ### 📍 Target Location
TARGET_LAT = 32.672190261170904 # @param {type:"number"}
TARGET_LNG = -96.80242361760473 # @param {type:"number"}

def get_spatial_imagery(target_lat, target_lng):
    print(f"📡 Querying BigQuery for tracks near {target_lat}, {target_lng}...")
    query_tracks = f"""
    SELECT
        t.trackId, ST_X(t.geometry) as lng, ST_Y(t.geometry) as lat,
        t.observation0, t.observation1, t.observation2, t.observation3,
        t.observation4, t.observation5, t.observation6,
        o.cameraPose.headingDeg as vehicle_heading
    FROM `{PROJECT_ID}.{DATASET_ID}.tracks_unnested` t
    JOIN `{PROJECT_ID}.{DATASET_ID}.observations` o ON t.observation0 = o.observationId
    ORDER BY ST_DISTANCE(t.geometry, ST_GEOGPOINT({target_lng}, {target_lat})) ASC
    LIMIT 10
    """
    tracks = list(client_bq.query(query_tracks).result())
    if not tracks: return []

    track_context = []
    for idx, track in enumerate(tracks):
        bearing = calculate_bearing((track.lat, track.lng), (target_lat, target_lng))
        for cam_id in range(7):
            obs_id = getattr(track, f"observation{cam_id}")
            if obs_id:
                track_context.append({
                    "track_id": track.trackId, "obs_id": obs_id, "cam_id": cam_id,
                    "v_lat": track.lat, "v_lng": track.lng, "target_bearing": bearing, "seq_idx": idx
                })

    obs_ids_str = ",".join([f"'{x['obs_id']}'" for x in track_context])
    query_details = f"""
    SELECT o.observationId, o.cameraPose.pitchDeg as pitch, o.cameraPose.headingDeg as obs_heading, u.signedUrl
    FROM `{PROJECT_ID}.{DATASET_ID}.observations` o
    JOIN `{PROJECT_ID}.{DATASET_ID}.urls_new` u ON o.observationId = u.observationId
    WHERE o.observationId IN ({obs_ids_str})
    """
    details_map = {row.observationId: row for row in client_bq.query(query_details).result()}
    view_names = {0:"front_right", 1:"right", 2:"rear_right", 3:"rear_left", 4:"left", 5:"front_left", 6:"zenith"}
    results = []

    for item in track_context:
        obs_id = item["obs_id"]
        if obs_id in details_map:
            d = details_map[obs_id]
            dist = geopy.distance.geodesic((item["v_lat"], item["v_lng"]), (target_lat, target_lng)).meters
            if d.pitch > 15 and dist < 20: continue
            if not is_within_fov(d.obs_heading, item["target_bearing"]): continue
            gs_link = d.signedUrl.replace("https://storage.mtls.cloud.google.com/", "gs://")
            results.append({
                "metadata": {"observation_id": obs_id, "spatial_context": {"heading": d.obs_heading, "view": view_names.get(item["cam_id"], "unknown"), "gps": [item["v_lat"], item["v_lng"],], "dist": round(dist, 2)}},
                "image_link": d.signedUrl, "gs_link": gs_link
            })
    return results

imagery_results = get_spatial_imagery(TARGET_LAT, TARGET_LNG)

📡 Querying BigQuery for tracks near 32.672190261170904, -96.80242361760473...


### 🧠 Step 4: Gemini Discovery & Clustering
Finally, we use **Gemini 3 Pro** to select the best vantage points from the metadata, and **Gemini 3 Flash** to analyze the actual images. The AI identifies residential features and groups images belonging to the same property using a generated 'Property ID'.

In [4]:
# @title 🧠 4. Gemini Discovery & Clustering
# @markdown Performs visual analysis and grouping using Gemini 3.5 models.

# @markdown ### Model & UI Settings
MODEL_PRO = "gemini-3.5-flash" # @param {type:"string"}
MODEL_FLASH = "gemini-3.5-flash" # @param {type:"string"}
MEDIA_RESOLUTION = "MEDIA_RESOLUTION_LOW" # @param ["MEDIA_RESOLUTION_LOW", "MEDIA_RESOLUTION_MEDIUM", "MEDIA_RESOLUTION_HIGH"]
MAX_RETURN_SAMPLES = 15 # @param {type:"slider", min:1, max:25, step:1}

def run_image_discovery(all_images, target_lat, target_lng):
    if not all_images:
        print("No imagery to process.")
        return

    print(f"📊 Discovery Agent ({MODEL_PRO}): Selecting optimal vantage points...")
    meta_prompt = f"Goal: Retrieve IDs that provide a direct view of {target_lat}, {target_lng}. Return JSON list of strings."
    meta_payload = [{"id": x['metadata']['observation_id'], "dist": x['metadata']['spatial_context']['dist'], "view": x['metadata']['spatial_context']['view']} for x in all_images]

    try:
        resp_meta = client_genai.models.generate_content(model=MODEL_PRO, contents=[meta_prompt, json.dumps(meta_payload)], config=types.GenerateContentConfig(response_mime_type="application/json"))
        tracker.add_usage(MODEL_PRO, resp_meta.usage_metadata)
        selected_ids = json.loads(resp_meta.text)
        # Robust handling for different JSON structures (list or dict with list)
        if isinstance(selected_ids, dict):
            for val in selected_ids.values():
                if isinstance(val, list):
                    selected_ids = val
                    break
    except Exception as e:
        print(f"Selection failed: {e}"); return

    property_clusters = {}
    for obs_id in selected_ids:
        img_obj = next((x for x in all_images if x['metadata']['observation_id'] == obs_id), None)
        if not img_obj: continue
        try:
            cluster_prompt = "Identify residential structure features and assign a Property ID. Return JSON: {'visible': bool, 'property_id': str, 'signature_bullet_points': [str], 'reason': str}"
            image_part = types.Part.from_uri(file_uri=img_obj['gs_link'], mime_type="image/jpeg")
            resp_val = client_genai.models.generate_content(model=MODEL_FLASH, contents=[image_part, cluster_prompt], config=types.GenerateContentConfig(response_mime_type="application/json", media_resolution=MEDIA_RESOLUTION))
            tracker.add_usage(MODEL_FLASH, resp_val.usage_metadata)
            analysis = json.loads(resp_val.text)

            # Handle cases where analysis might be a list containing one dict
            if isinstance(analysis, list) and len(analysis) > 0:
                analysis = analysis[0]

            if isinstance(analysis, dict) and analysis.get('visible'):
                prop_id = analysis.get('property_id', 'Unknown Property')
                if prop_id not in property_clusters:
                    property_clusters[prop_id] = {"images": [], "signature": analysis.get('signature_bullet_points', [])}
                property_clusters[prop_id]["images"].append(img_obj)
        except Exception as e: print(f"Error processing {obs_id}: {e}")

    # Display Results
    for prop_id, data in property_clusters.items():
        print(f"🏠 CLUSTER: {prop_id}")
        for res in data['images'][:MAX_RETURN_SAMPLES]:
            display(IPImage(url=res['image_link'], width=500))
    tracker.print_summary()

run_image_discovery(imagery_results, TARGET_LAT, TARGET_LNG)

📊 Discovery Agent (gemini-3.5-flash): Selecting optimal vantage points...


🏠 CLUSTER: 1


🏠 CLUSTER: Property_1


🏠 CLUSTER: Prop_1


🏠 CLUSTER: prop_0


...

🏠 CLUSTER: P_1


🏠 CLUSTER: P1


🏠 CLUSTER: P01


🏠 CLUSTER: P-1



💰 VERTEX AI & GEMINI USAGE SUMMARY
Total Input Tokens:  7,034
Total Output Tokens: 4,850
Total Estimated Cost: $0.0181 USD
